# ACIS Insurance Risk Analytics: Exploratory Data Analysis

This notebook performs the first-stage exploratory data analysis for the AlphaCare Insurance Solutions (ACIS) insurance risk analytics challenge.

The goal is to understand data quality, claim risk, profitability, vehicle patterns, and customer/geographic segments before hypothesis testing and modeling.

## 1. Setup

Import the project utilities and plotting libraries. The notebook expects the dataset path to be updated in the `DATA_PATH` variable.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_loader import (
    duplicate_row_summary,
    inspect_column_types,
    load_insurance_data,
    missing_value_summary,
    summarize_dataset,
)
from src.eda_utils import (
    add_risk_metrics,
    calculate_loss_ratio,
    calculate_margin,
    group_by_gender,
    group_by_province,
    group_by_vehicle_type,
    group_risk_summary,
)

sns.set_theme(style="whitegrid", palette="viridis")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.4f}".format)

## 2. Load the Dataset

Update `DATA_PATH` to point to the DVC-tracked insurance dataset in the `data/` directory. The loader supports CSV, TXT, Excel, Parquet, and JSON files.

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "insurance_data.csv"

# If the file is pipe-delimited or semicolon-delimited, pass sep="|" or sep=";".
df = load_insurance_data(DATA_PATH)

df.head()

**Business interpretation:** Confirm that the dataset loaded successfully and that the visible columns match the expected ACIS insurance policy, customer, vehicle, premium, and claim fields. If the file path or delimiter is wrong, fix it before continuing.

## 3. Dataset Summary and Quality Checks

Summarize the dataset size, missing values, data types, and duplicate records before deeper analysis.

In [ ]:
summary = summarize_dataset(df)
summary

In [ ]:
column_types = inspect_column_types(df)
column_types.sort_values("missing_percentage", ascending=False).head(20)

In [ ]:
missing_summary = missing_value_summary(df)
missing_summary[missing_summary["missing_count"] > 0].head(30)

In [ ]:
duplicate_row_summary(df)

**Business interpretation:** Missing values can reduce confidence in segment-level findings, especially if important fields such as claims, premium, province, gender, or vehicle attributes are incomplete. Duplicate records may overstate exposure, premiums, and claims, so they should be investigated before final reporting.

## 4. Descriptive Statistics

Review the distribution of numeric variables, especially premium, claims, and vehicle valuation fields.

In [ ]:
numeric_columns = df.select_dtypes(include="number").columns.tolist()
df[numeric_columns].describe().T

In [ ]:
key_numeric_cols = [
    col for col in ["TotalPremium", "TotalClaims", "CustomValueEstimate"] if col in df.columns
]

df[key_numeric_cols].describe().T

**Business interpretation:** Premium, claim, and vehicle-value distributions show the scale of exposure and claim severity. Large gaps between means and medians may indicate skewed risk, high-value vehicles, or extreme claim events that need careful modeling.

## 5. Loss Ratio and Margin

Loss ratio measures claims relative to premium. Margin measures premium minus claims. Together, they show whether a segment is profitable or loss-making.

In [ ]:
df_risk = add_risk_metrics(df)

overall_loss_ratio = calculate_loss_ratio(df_risk)
overall_margin = calculate_margin(df_risk)

pd.DataFrame(
    {
        "metric": ["overall_loss_ratio", "overall_margin"],
        "value": [overall_loss_ratio, overall_margin],
    }
)

**Business interpretation:** A high loss ratio means claims consume a large share of premiums and may signal underpricing or high-risk segments. Positive margin suggests premium adequacy, while negative margin requires pricing, underwriting, or marketing attention.

## 6. Visualization 1: Premium and Claim Distributions

Use histograms to understand skewness and concentration in premiums and claims.

In [ ]:
fig, axes = plt.subplots(1, len(key_numeric_cols), figsize=(6 * len(key_numeric_cols), 4))
if len(key_numeric_cols) == 1:
    axes = [axes]

for ax, column in zip(axes, key_numeric_cols):
    sns.histplot(df_risk[column].dropna(), kde=True, bins=40, ax=ax)
    ax.set_title(f"Distribution of {column}")
    ax.set_xlabel(column)

plt.tight_layout()
plt.show()

**Business interpretation:** Skewed premium or claim distributions indicate that a small number of policies may drive a large share of financial risk. This is important for severity modeling because extreme claims can dominate model behavior.

## 7. Visualization 2: Outlier Detection with Box Plots

Box plots help identify extreme values in premiums, claims, and vehicle valuations.

In [ ]:
plt.figure(figsize=(12, 5))
sns.boxplot(data=df_risk[key_numeric_cols], orient="h")
plt.title("Outlier Check for Key Numeric Variables")
plt.xlabel("Value")
plt.show()

**Business interpretation:** Outliers should not be removed automatically because they may represent genuine high-severity claims or high-value vehicles. They should be validated and handled carefully during modeling.

## 8. Visualization 3: Correlation Heatmap

A correlation heatmap helps identify numeric features related to premium, claims, margin, and vehicle value.

In [ ]:
corr_columns = df_risk.select_dtypes(include="number").columns.tolist()
correlation_matrix = df_risk[corr_columns].corr(numeric_only=True)

plt.figure(figsize=(12, 9))
sns.heatmap(correlation_matrix, cmap="coolwarm", center=0, linewidths=0.5)
plt.title("Correlation Heatmap of Numeric Variables")
plt.show()

**Business interpretation:** Strong correlations with claims or margin can suggest useful pricing and modeling features. Very high correlations between predictor variables may also indicate redundancy that should be considered during model building.

## 9. Segment Analysis by Province, Vehicle Type, and Gender

Compare loss ratio and margin across business-relevant groups to identify low-risk and high-risk segments.

In [ ]:
province_summary = group_by_province(df_risk)
province_summary.head(15)

In [ ]:
if "Province" in df_risk.columns:
    top_provinces = province_summary.sort_values("policy_count", ascending=False).head(10)

    plt.figure(figsize=(12, 5))
    sns.barplot(data=top_provinces, x="Province", y="loss_ratio")
    plt.title("Loss Ratio by Province")
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Loss Ratio")
    plt.show()

**Business interpretation:** Provinces with lower loss ratios and positive margins may be attractive for targeted growth. Provinces with high loss ratios may require pricing review, underwriting changes, or deeper investigation of claim drivers.

In [ ]:
vehicle_type_summary = group_by_vehicle_type(df_risk)
vehicle_type_summary.head(15)

In [ ]:
if "VehicleType" in df_risk.columns:
    top_vehicle_types = vehicle_type_summary.sort_values("policy_count", ascending=False).head(10)

    plt.figure(figsize=(12, 5))
    sns.barplot(data=top_vehicle_types, x="VehicleType", y="loss_ratio")
    plt.title("Loss Ratio by Vehicle Type")
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Loss Ratio")
    plt.show()

**Business interpretation:** Vehicle types with consistently high loss ratios may reflect higher repair costs, usage patterns, or theft/accident exposure. Low-loss vehicle types can support safer pricing and marketing strategies.

In [ ]:
gender_summary = group_by_gender(df_risk)
gender_summary

In [ ]:
if "Gender" in df_risk.columns:
    plt.figure(figsize=(8, 5))
    sns.barplot(data=gender_summary, x="Gender", y="loss_ratio")
    plt.title("Loss Ratio by Gender")
    plt.ylabel("Loss Ratio")
    plt.show()

**Business interpretation:** Gender-level differences should be interpreted carefully and validated with hypothesis testing. Observed EDA differences may be influenced by exposure mix, vehicle types, location, and policy features.

## 10. Car Make and Model Analysis

Compare claim risk and margin across vehicle makes and models to identify vehicles associated with higher or lower insurance risk.

In [ ]:
make_col = "make" if "make" in df_risk.columns else "Make" if "Make" in df_risk.columns else None
model_col = "Model" if "Model" in df_risk.columns else "model" if "model" in df_risk.columns else None

if make_col:
    make_summary = group_risk_summary(df_risk, make_col)
    display(make_summary.head(15))
else:
    print("No car make column found. Expected 'Make' or 'make'.")

In [ ]:
if model_col:
    model_summary = group_risk_summary(df_risk, model_col)
    display(model_summary.head(15))
else:
    print("No car model column found. Expected 'Model' or 'model'.")

In [ ]:
if make_col:
    top_makes = make_summary.sort_values("policy_count", ascending=False).head(15)

    plt.figure(figsize=(12, 5))
    sns.barplot(data=top_makes, x=make_col, y="loss_ratio")
    plt.title("Loss Ratio by Car Make")
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Loss Ratio")
    plt.show()

**Business interpretation:** Makes and models with high loss ratios may require adjusted pricing, tighter underwriting, or deeper inspection of repair cost and claim frequency. Low-risk vehicle groups may support more competitive premiums or targeted marketing.

## 11. Key Takeaways

Use this section to summarize the most important EDA findings before moving into hypothesis testing.

Suggested points to capture:

- Overall data quality issues and whether any fields require cleaning.
- Overall loss ratio and margin.
- Highest-risk and lowest-risk provinces.
- Vehicle types, makes, or models with unusual claim patterns.
- Any gender-level patterns that should be tested statistically.
- Outliers or skewed variables that may affect modeling.